# Prepare audio data for image recognition

The data is pretty good, but there's a few samples that aren't exactly 1 second long and some samples that are either truncated or don't contain very much of the word.

The code in the notebook attempts to filter out the broken audio so that we are only using good audio.

We then generate spectrograms of each word. We mix in background noise with the words to make it a more realistic audio sample.

## Download data set
Download from: https://storage.cloud.google.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz - approx 2.3 GB

And then run
```
tar -xzf data_speech_commands_v0.02.tar.gz -C speech_data
```

In [1]:
import tensorflow as tf

path = tf.keras.utils.get_file(
    fname="speech_commands_v0.02.tar.gz",
    origin="http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz",
    cache_dir="/content",
    cache_subdir=""
)

print("下載完成：", path)

2428923189/2428923189 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step
下載完成： /content/speech_commands_v0.02.tar.gz


In [2]:
import tarfile
import os

os.makedirs("/content/speech_data", exist_ok=True)

with tarfile.open("/content/speech_commands_v0.02.tar.gz", "r:gz") as tar:
    tar.extractall("/content/speech_data")

print("解壓完成")
print(os.listdir("/content/speech_data")[:30])

/tmp/ipykernel_3067/2420850436.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/content/speech_data")


解壓完成
['no', 'happy', '.DS_Store', 'visual', 'house', 'right', 'backward', 'seven', 'up', 'forward', 'wow', 'one', 'bird', 'five', 'yes', 'follow', 'on', 'README.md', 'tree', 'marvin', 'stop', 'four', 'off', 'six', 'testing_list.txt', 'three', 'down', 'LICENSE', 'go', 'left']


In [3]:
!file speech_commands_v0.02.tar.gz

speech_commands_v0.02.tar.gz: gzip compressed data, last modified: Wed Apr 11 17:56:01 2018, from Unix, original size modulo 2^32 3417661440


In [4]:
!pip install tensorflow-io

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 82.2 MB/s eta 0:00:00


In [5]:
import tensorflow as tf
import numpy as np
from tensorflow.io import gfile
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [6]:
SPEECH_DATA='/content/speech_data'

In [7]:
# The audio is all sampled at 16KHz and should all be 1 second in length - so 1 second is 16000 samples
EXPECTED_SAMPLES=16000
# Noise floor to detect if any audio is present
NOISE_FLOOR=0.1
# How many samples should be abover the noise floor?
MINIMUM_VOICE_LENGTH=EXPECTED_SAMPLES/4

In [8]:
# list of folders we want to process in the speech_data folder
from tensorflow.python.ops import gen_audio_ops as audio_ops
words = [
    'backward',
    'bed',
    'bird',
    'cat',
    'dog',
    'down',
    'eight',
    'five',
    'follow',
    'forward',
    'four',
    'go',
    'happy',
    'house',
    'learn',
    'left',
    'marvin',
    'nine',
    'no',
    'off',
    'on',
    'one',
    'right',
    'seven',
    'sheila',
    'six',
    'stop',
    'three',
    'tree',
    'two',
    'up',
    'visual',
    'wow',
    'yes',
    'zero',
    '_background',
]

In [9]:
# get all the files in a directory
def get_files(word):
    return gfile.glob(SPEECH_DATA + '/'+word+'/*.wav')

# get the location of the voice
def get_voice_position(audio, noise_floor):
    audio = audio - np.mean(audio)
    audio = audio / np.max(np.abs(audio))
    return tfio.audio.trim(audio, axis=0, epsilon=noise_floor)

# Work out how much of the audio file is actually voice
def get_voice_length(audio, noise_floor):
    position = get_voice_position(audio, noise_floor)
    return (position[1] - position[0]).numpy()

# is enough voice present?
def is_voice_present(audio, noise_floor, required_length):
    voice_length = get_voice_length(audio, noise_floor)
    return voice_length >= required_length

# is the audio the correct length?
def is_correct_length(audio, expected_length):
    return (audio.shape[0]==expected_length).numpy()


def is_valid_file(file_name):
    try:
        audio_binary = tf.io.read_file(file_name)
        audio, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1)
        audio = tf.squeeze(audio, axis=-1)

        return len(audio) >= EXPECTED_SAMPLES

    except:
        return False


In [10]:
def get_spectrogram(audio):
    # normalise the audio
    audio = audio - np.mean(audio)
    audio = audio / np.max(np.abs(audio))
    # create the spectrogram
    spectrogram = audio_ops.audio_spectrogram(audio,
                                              window_size=320,
                                              stride=160,
                                              magnitude_squared=True).numpy()
    # reduce the number of frequency bins in our spectrogram to a more sensible level
    spectrogram = tf.nn.pool(
        input=tf.expand_dims(spectrogram, -1),
        window_shape=[1, 6],
        strides=[1, 6],
        pooling_type='AVG',
        padding='SAME')
    spectrogram = tf.squeeze(spectrogram, axis=0)
    spectrogram = np.log10(spectrogram + 1e-6)
    return spectrogram

In [11]:
# process a file into its spectrogram
def process_file(file_path):
    audio_binary = tf.io.read_file(file_path)
    audio, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)

    # 固定長度 16000
    audio = audio[:16000]
    zero_padding = tf.zeros([16000] - tf.shape(audio), dtype=tf.float32)
    audio = tf.cast(audio, tf.float32)
    audio = tf.concat([audio, zero_padding], 0)

    # spectrogram
    spectrogram = tf.signal.stft(
        audio,
        frame_length=255,
        frame_step=128
    )
    spectrogram = tf.abs(spectrogram)
    spectrogram = spectrogram[..., tf.newaxis]

    return spectrogram

In [12]:
train = []
validate = []
test = []

TRAIN_SIZE=0.8
VALIDATION_SIZE=0.1
TEST_SIZE=0.1

In [13]:
def process_files(file_names, label, word_name, repeat=1):
    # 用 Python list 重複，比 tf.repeat 更穩
    file_names = list(file_names) * repeat

    results = []
    for file_name in tqdm(file_names, desc=f"{word_name} ({label})", leave=False, total=len(file_names)):
        results.append((process_file(file_name), label))
    return results


# process the files for a word into the spectrogram and one hot encoding word value
def process_word(word, repeat=1):
    # the index of the word we are processing
    label = words.index(word)

    # get a list of file names for the word
    file_names = [
        file_name
        for file_name in tqdm(get_files(word), desc=f"Checking {word}", leave=False)
        if is_valid_file(file_name)
    ]

    # randomly shuffle the filenames
    np.random.shuffle(file_names)

    # split the files into train, validate and test buckets
    train_size = int(TRAIN_SIZE * len(file_names))
    validation_size = int(VALIDATION_SIZE * len(file_names))
    test_size = int(TEST_SIZE * len(file_names))

    # get the training samples
    train.extend(
        process_files(
            file_names[:train_size],
            label,
            word,
            repeat=repeat
        )
    )

    # and the validation samples
    validate.extend(
        process_files(
            file_names[train_size:train_size + validation_size],
            label,
            word,
            repeat=repeat
        )
    )

    # and the test samples
    test.extend(
        process_files(
            file_names[train_size + validation_size:train_size + validation_size + test_size],
            label,
            word,
            repeat=repeat
        )
    )


# process all the words and all the files
for word in tqdm(words, desc="Processing words"):
    if '_' not in word:
        # add more examples of happy to balance our training set
        repeat = 50 if word == 'forward' else 1
        process_word(word, repeat=repeat)

print(len(train), len(test), len(validate))

Processing words:   0%|          | 0/36 [00:00<?, ?it/s]

Checking backward:   0%|          | 0/1664 [00:00<?, ?it/s]

backward (0):   0%|          | 0/1246 [00:00<?, ?it/s]

backward (0):   0%|          | 0/155 [00:00<?, ?it/s]

backward (0):   0%|          | 0/155 [00:00<?, ?it/s]

...

Checking zero:   0%|          | 0/4052 [00:00<?, ?it/s]

zero (34):   0%|          | 0/3004 [00:00<?, ?it/s]

zero (34):   0%|          | 0/375 [00:00<?, ?it/s]

zero (34):   0%|          | 0/375 [00:00<?, ?it/s]

133189 16627 16627


In [15]:
import tensorflow_io as tfio

/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl5mutex6unlockEv']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: libtensorflow_io.so, from paths: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so']
caused by: ['/usr/local/lib/python3.12/dist-packages/tensorflow_io/python/ops/libtensorflow_io.so: undefined symbol: _ZN3tsl7strings13safe_strtou64ESt17basic_string_viewIcSt11char_traitsIcEEPm']
  warnings.warn(

In [18]:
# process the background noise files
def process_background(file_name, label):
    # load the audio file without tensorflow-io
    audio_binary = tf.io.read_file(file_name)
    audio, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio = tf.cast(audio, tf.float32).numpy()

    audio_length = len(audio)
    samples = []

    for section_start in tqdm(range(0, audio_length - EXPECTED_SAMPLES, 8000), desc=file_name, leave=False):
        section_end = section_start + EXPECTED_SAMPLES
        section = audio[section_start:section_end]
        section = np.reshape(section, (EXPECTED_SAMPLES, 1))

        spectrogram = get_spectrogram(section)
        samples.append((spectrogram, label))

    # simulate random utterances
    for section_index in tqdm(range(1000), desc="Simulated Words", leave=False):
        section_start = np.random.randint(0, audio_length - EXPECTED_SAMPLES)
        section_end = section_start + EXPECTED_SAMPLES
        section = audio[section_start:section_end]

        result = np.zeros(EXPECTED_SAMPLES)

        voice_length = np.random.randint(MINIMUM_VOICE_LENGTH // 2, EXPECTED_SAMPLES)
        voice_start = np.random.randint(0, EXPECTED_SAMPLES - voice_length)
        hamming = np.hamming(voice_length)

        result[voice_start:voice_start + voice_length] = hamming * section[voice_start:voice_start + voice_length]
        result = np.reshape(result, (EXPECTED_SAMPLES, 1))

        spectrogram = get_spectrogram(result)
        samples.append((spectrogram, label))

    np.random.shuffle(samples)

    train_size = int(TRAIN_SIZE * len(samples))
    validation_size = int(VALIDATION_SIZE * len(samples))
    test_size = int(TEST_SIZE * len(samples))

    train.extend(samples[:train_size])
    validate.extend(samples[train_size:train_size + validation_size])
    test.extend(samples[train_size + validation_size:])

for file_name in tqdm(get_files('_background_noise_'), desc="Processing Background Noise"):
    process_background(file_name, words.index("_background"))

print(len(train), len(test), len(validate))

Processing Background Noise:   0%|          | 0/6 [00:00<?, ?it/s]

/content/speech_data/_background_noise_/white_noise.wav:   0%|          | 0/118 [00:00<?, ?it/s]

Simulated Words:   0%|          | 0/1000 [00:00<?, ?it/s]

/content/speech_data/_background_noise_/exercise_bike.wav:   0%|          | 0/121 [00:00<?, ?it/s]

Simulated Words:   0%|          | 0/1000 [00:00<?, ?it/s]

...

/content/speech_data/_background_noise_/running_tap.wav:   0%|          | 0/121 [00:00<?, ?it/s]

Simulated Words:   0%|          | 0/1000 [00:00<?, ?it/s]

/content/speech_data/_background_noise_/doing_the_dishes.wav:   0%|          | 0/189 [00:00<?, ?it/s]

Simulated Words:   0%|          | 0/1000 [00:00<?, ?it/s]

138617 17312 17303


In [19]:
def process_problem_noise(file_name, label):
    samples = []

    # load audio file
    audio_binary = tf.io.read_file(file_name)
    audio, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio = tf.cast(audio, tf.float32).numpy()

    audio_length = len(audio)

    for section_start in tqdm(range(0, audio_length - EXPECTED_SAMPLES, 400), desc=file_name, leave=False):
        section_end = section_start + EXPECTED_SAMPLES
        section = audio[section_start:section_end]
        section = np.reshape(section, (EXPECTED_SAMPLES, 1))

        spectrogram = get_spectrogram(section)
        samples.append((spectrogram, label))

    np.random.shuffle(samples)

    train_size = int(TRAIN_SIZE * len(samples))
    validation_size = int(VALIDATION_SIZE * len(samples))
    test_size = int(TEST_SIZE * len(samples))

    train.extend(samples[:train_size])
    validate.extend(samples[train_size:train_size + validation_size])
    test.extend(samples[train_size + validation_size:])

for file_name in tqdm(get_files("_problem_noise_"), desc="Processing problem noise"):
    process_problem_noise(file_name, words.index("_background"))

print(len(train), len(test), len(validate))

Processing problem noise: 0it [00:00, ?it/s]

138617 17312 17303


In [20]:
def process_mar_sounds(file_name, label):
    samples = []

    # load the audio file
    audio_binary = tf.io.read_file(file_name)
    audio, sample_rate = tf.audio.decode_wav(audio_binary, desired_channels=1)
    audio = tf.squeeze(audio, axis=-1)
    audio = tf.cast(audio, tf.float32).numpy()

    audio_length = len(audio)

    for section_start in tqdm(range(0, audio_length - EXPECTED_SAMPLES, 4000), desc=file_name, leave=False):
        section_end = section_start + EXPECTED_SAMPLES
        section = audio[section_start:section_end]

        section = section - np.mean(section)
        section = section / np.max(np.abs(section))

        # add some random background noise
        background_volume = np.random.uniform(0, 0.1)

        background_files = get_files('_background_noise_')
        background_file = np.random.choice(background_files)

        background_binary = tf.io.read_file(background_file)
        background, sample_rate = tf.audio.decode_wav(background_binary, desired_channels=1)
        background = tf.squeeze(background, axis=-1)
        background = tf.cast(background, tf.float32).numpy()

        background_start = np.random.randint(0, len(background) - EXPECTED_SAMPLES)
        background = background[background_start:background_start + EXPECTED_SAMPLES]

        background = background - np.mean(background)
        background = background / np.max(np.abs(background))

        section = section + background_volume * background

        section = np.reshape(section, (EXPECTED_SAMPLES, 1))

        spectrogram = get_spectrogram(section)
        samples.append((spectrogram, label))

    np.random.shuffle(samples)

    train_size = int(TRAIN_SIZE * len(samples))
    validation_size = int(VALIDATION_SIZE * len(samples))
    test_size = int(TEST_SIZE * len(samples))

    train.extend(samples[:train_size])
    validate.extend(samples[train_size:train_size + validation_size])
    test.extend(samples[train_size + validation_size:])

for file_name in tqdm(get_files("_mar_sounds_"), desc="Processing problem noise"):
    process_mar_sounds(file_name, words.index("_background"))

print(len(train), len(test), len(validate))

Processing problem noise: 0it [00:00, ?it/s]

138617 17312 17303


In [21]:
print(len(train), len(test), len(validate))

138617 17312 17303


In [22]:
# randomise the training samples
np.random.shuffle(train)

In [23]:
X_train, Y_train = zip(*train)
X_validate, Y_validate = zip(*validate)
X_test, Y_test = zip(*test)

In [26]:
from collections import Counter
import numpy as np

train_shapes = Counter([np.array(x).shape for x in X_train])
validate_shapes = Counter([np.array(x).shape for x in X_validate])
test_shapes = Counter([np.array(x).shape for x in X_test])

print("Train shapes:", train_shapes)
print("Validate shapes:", validate_shapes)
print("Test shapes:", test_shapes)

Train shapes: Counter({(124, 129, 1): 133189, (99, 43, 1): 5428})
Validate shapes: Counter({(124, 129, 1): 16627, (99, 43, 1): 676})
Test shapes: Counter({(124, 129, 1): 16627, (99, 43, 1): 685})


In [27]:
TARGET_SHAPE = (124, 129, 1)

# ---------- train ----------
filtered_train = [
    (x, y) for x, y in zip(X_train, Y_train)
    if np.array(x).shape == TARGET_SHAPE
]

X_train_fixed = np.array([x for x, y in filtered_train], dtype=np.float32)
Y_train_fixed = np.array([y for x, y in filtered_train])

# ---------- validate ----------
filtered_validate = [
    (x, y) for x, y in zip(X_validate, Y_validate)
    if np.array(x).shape == TARGET_SHAPE
]

X_validate_fixed = np.array([x for x, y in filtered_validate], dtype=np.float32)
Y_validate_fixed = np.array([y for x, y in filtered_validate])

# ---------- test ----------
filtered_test = [
    (x, y) for x, y in zip(X_test, Y_test)
    if np.array(x).shape == TARGET_SHAPE
]

X_test_fixed = np.array([x for x, y in filtered_test], dtype=np.float32)
Y_test_fixed = np.array([y for x, y in filtered_test])

print("Train:", X_train_fixed.shape, Y_train_fixed.shape)
print("Validate:", X_validate_fixed.shape, Y_validate_fixed.shape)
print("Test:", X_test_fixed.shape, Y_test_fixed.shape)

# ---------- save ----------
np.savez_compressed(
    "training_spectrogram.npz",
    X=X_train_fixed, Y=Y_train_fixed
)
print("Saved training data")

np.savez_compressed(
    "validation_spectrogram.npz",
    X=X_validate_fixed, Y=Y_validate_fixed
)
print("Saved validation data")

np.savez_compressed(
    "test_spectrogram.npz",
    X=X_test_fixed, Y=Y_test_fixed
)
print("Saved test data")

Train: (133189, 124, 129, 1) (133189,)
Validate: (16627, 124, 129, 1) (16627,)
Test: (16627, 124, 129, 1) (16627,)
Saved training data
Saved validation data
Saved test data


In [ ]:
# get the width and height of the spectrogram "image"
IMG_WIDTH=X_train[0].shape[0]
IMG_HEIGHT=X_train[0].shape[1]

In [ ]:
def plot_images2(images_arr, imageWidth, imageHeight):
    fig, axes = plt.subplots(5, 5, figsize=(10, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        ax.imshow(np.reshape(img, (imageWidth, imageHeight)))
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
word_index = words.index("forward")

X_forward = np.array(X_train)[np.array(Y_train) == word_index]
Y_forward = np.array(Y_train)[np.array(Y_train) == word_index]
plot_images2(X_forward[:20], IMG_WIDTH, IMG_HEIGHT)
print(Y_forward[:20])

In [ ]:
word_index = words.index("yes")

X_yes = np.array(X_train)[np.array(Y_train) == word_index]
Y_yes = np.array(Y_train)[np.array(Y_train) == word_index]
plot_images2(X_yes[:20], IMG_WIDTH, IMG_HEIGHT)
print(Y_yes[:20])